## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import kagglehub
import os

from sklearn.ensemble import RandomForestClassifier    
from sklearn.model_selection import train_test_split    
from sklearn.metrics import accuracy_score    


## Load dataset

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
train_data.head()

In [ ]:
test_data.head()

## Missing Values Analysis


In [ ]:
# Function for analyzing missing values. 
# 1. which columns contain missing data;
# 2. how severe the problem is;
# 3. whether columns should be removed or filled.

def missing_value_table(df):
    miss_value = df.isnull().sum()    # Total number of missing values
    mis_value_percent = 100 * (df.isnull().sum() / len(df))    # Percentage of missing values
    mis_val_table = pd.concat([miss_value, mis_value_percent], axis = 1)    # Combine results into one table
    
        # Convenient display of values in a table
    mis_val_table_ren_col = mis_val_table.rename(
        columns = {0 : 'Missing values', 1 : '% of Total Values'})    # Rename columns for readability
    mis_val_table_ren_col = mis_val_table_ren_col[
        mis_val_table_ren_col.iloc[:, 1] != 0].sort_values(
        '% of Total Values', ascending = False).round(1)    # Sort by percentage of missing values
    print('Number of columns in the table: ' + str(df.shape[1]) + 
          '\nNumber of columns with gaps: ' + str(mis_val_table_ren_col.shape[0]))    # Output of general information
    return mis_val_table_ren_col

In [ ]:
missing_value_table(train_data)

In [ ]:
missing_value_table(test_data)

## Feature Engineering


In [ ]:
# Extract title from passenger name.
train_data['Title'] = (
    train_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False))
test_data['Title'] = (
    test_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
)

In [ ]:
train_data.head()

In [ ]:
train_data['Title'].value_counts()

In [ ]:
train_data.drop(['PassengerId', 'Name', 'Cabin', 'Ticket'], axis = 1, inplace = True)
test_data.drop(['Name', 'Cabin', 'Ticket'], axis = 1, inplace = True)

In [ ]:
# Convert gender categories into numbers.
sex = {'male' : 0, 'female' : 1}
train_data['Sex'] = train_data['Sex'].map(sex)
test_data['Sex'] = test_data['Sex'].map(sex)



In [ ]:
train_data['Embarked'].value_counts()

In [ ]:
test_data['Embarked'].value_counts()

In [ ]:
# Fill missing age values using median. Median is more robust to outliers

train_data['Age'] = train_data['Age'].fillna(train_data['Age'].median())
train_data['Embarked'] = train_data['Embarked'].fillna('S')

test_data['Age'] = test_data['Age'].fillna(test_data['Age'].median())
test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].median())
test_data['Embarked'] = test_data['Embarked'].fillna('S')

In [ ]:
# OHE - Convert categorical variables into numerical dummy columns.
train_data = pd.get_dummies(train_data, columns=['Embarked'], dtype = int)
test_data = pd.get_dummies(test_data, columns=['Embarked'], dtype = int)
train_data = pd.get_dummies(train_data, columns=['Title'], dtype = int)
test_data = pd.get_dummies(test_data, columns=['Title'], dtype = int)

In [ ]:
train_data.head()

In [ ]:
missing_value_table(train_data)

In [ ]:
missing_value_table(test_data)

In [ ]:
for columns in enumerate(train_data.columns):
    print(f"{columns}")

In [ ]:
for columns in enumerate(test_data.columns):
    print(f"{columns}")

In [ ]:
# Drop Unnecessary Columns
train_data.drop(['Title_Capt', 'Title_Col', 'Title_Countess', 'Title_Don', 'Title_Dr', 'Title_Jonkheer', 'Title_Sir', 'Title_Mme', 'Title_Ms', 'Title_Lady', 'Title_Major', 'Title_Mlle', 'Title_Rev'], axis = 1, inplace = True)
test_data.drop(['Title_Col', 'Title_Dona', 'Title_Dr', 'Title_Ms', 'Title_Rev'], axis = 1, inplace = True)

In [ ]:
# Create family size feature.
# Hypothesis:
# passengers traveling with family
# may have had different survival chances.

train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1

train_data['IsAlone'] = (train_data['FamilySize'] == 1).astype(int)
test_data['IsAlone'] = (test_data['FamilySize'] == 1).astype(int)

In [ ]:
# Drop Unnecessary Columns
train_data.drop(['SibSp', 'Parch'], axis = 1, inplace = True)
test_data.drop(['SibSp', 'Parch'], axis = 1, inplace = True)

In [ ]:
train_data.head()

## Prepare Features and Target

In [ ]:
y = train_data['Survived']
train_data.drop('Survived', axis = 1, inplace = True)   # Remove target column from features

test_PassengerId = test_data['PassengerId']

In [ ]:
test_data.drop('PassengerId', axis = 1, inplace = True)   # Drop Unnecessary Columns

# Align train and test columns.
train_data, test_data = train_data.align(
    test_data,
    join = 'left',
    axis = 1,
    fill_value = 0
)

# Features
X = train_data
X_test = test_data

## Train / Validation Split


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify=y
)


## Model Training


In [ ]:
model = RandomForestClassifier(
    n_estimators = 1000,
    max_depth = 4,
    min_samples_split = 4,
    min_samples_leaf = 1,
    max_features = 'sqrt',
    bootstrap = True,
    random_state = 42
)

In [ ]:
model.fit(X_train, y_train)

## Validation


In [ ]:
valid_predictions = model.predict(X_valid)
accuracy = accuracy_score(
    y_valid,
    valid_predictions
)
print('Accuracy: ', round(accuracy * 100, 2))

## Feature Importance

In [ ]:
importance= pd.DataFrame({
    'Features' : X.columns,
    'Importance' : model.feature_importances_
})
print(importance.sort_values('Importance', ascending = False))

In [ ]:
print(X_test.columns)

In [ ]:
print(X.columns)

## Predict Test Dataset

In [ ]:
test_predictions = model.predict(X_test)

In [ ]:
# Create Submission File
output = pd.DataFrame({
    'PassengerId' : test_PassengerId,
    'Survived' : test_predictions})
output.to_csv('/kaggle/working/submission.csv', index=False)
print ('Your permission saved!')